[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/01_differential_privacy/01_differential_privacy.ipynb)

# 01 · 差分隐私（用 numpy 从零实现）

目标：把 **(ε,δ)-DP**、**敏感度**、**Laplace / Gaussian 机制**、**组合定理**、**隐私-效用权衡** 用 numpy 实现，并用 `assert` + 经验验证钉死。

路线：敏感度 → Laplace 机制(+经验 ε 验证) → Gaussian 机制 → 组合(串/并) → 隐私-效用曲线 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**DP 机制 = 真值 + 按敏感度/ε 标定的噪声**；**ε = 可消费的隐私预算**；**对拍 = 验证理论期望/方差 + 经验 ε ≤ e^ε**。

## 1 · 敏感度：单个人最多能改变查询多少

敏感度 = 在相邻数据集（差一条记录）上查询结果的最大变化。它决定要加多少噪声。

我们对几个经典查询**经验地**估计敏感度：枚举「去掉任意一条记录」，看结果变化的最大值。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def empirical_l1_sensitivity(query, data):
    '''经验 L1 敏感度：去掉任意一条记录，结果变化的最大 L1 范数。'''
    base = np.atleast_1d(query(data))
    max_change = 0.0
    for i in range(len(data)):
        neighbor = np.delete(data, i, axis=0)
        change = np.sum(np.abs(np.atleast_1d(query(neighbor)) - base))
        max_change = max(max_change, change)
    return max_change

# 计数查询：有多少条记录 -> 删一条变化恰好 1
count_q = lambda d: float(len(d))
data = rng.integers(0, 2, size=50).astype(float)   # 50 个 0/1
s_count = empirical_l1_sensitivity(count_q, data)
print(f'计数查询 敏感度 = {s_count:.3f}  (理论=1)')
assert abs(s_count - 1.0) < 1e-9, '计数敏感度应为 1'
print('✅ 计数查询敏感度=1：增删一个人，计数最多变 1')

再看**有界求和**：年龄被裁剪到 `[0, 120]`，单人最多贡献 120，所以求和的敏感度=120。

**关键**：裁剪取值范围是控制敏感度的核心手段——这正是模块 02「梯度裁剪」的前身。

In [ ]:
CLIP = 120.0
def bounded_sum(d):
    return float(np.sum(np.clip(d, 0, CLIP)))

ages = rng.uniform(0, 120, size=50)
s_sum = empirical_l1_sensitivity(bounded_sum, ages)
print(f'有界求和 敏感度 = {s_sum:.2f}  (理论上界=CLIP=120)')
assert s_sum <= CLIP + 1e-6, '有界求和敏感度不应超过裁剪上界'
# 构造最坏情况：含一个贴边的样本，删它变化≈120
ages_worst = np.append(rng.uniform(0, 1, 49), 120.0)
s_worst = empirical_l1_sensitivity(bounded_sum, ages_worst)
print(f'最坏情况 敏感度 = {s_worst:.2f}  (接近 120)')
assert s_worst > 100, '含贴边样本时敏感度应接近裁剪上界'
print('✅ 求和敏感度由裁剪上界 CLIP 决定 —— 想降噪声就收紧 CLIP')

## 2 · Laplace 机制：纯 ε-DP

给查询加均值 0、尺度 `b = 敏感度/ε` 的 Laplace 噪声。我们实现它，并验证两件事：
① **无偏**（期望=真值）；② **噪声尺度正确**（Laplace 方差=2b²）。

In [ ]:
def laplace_mechanism(true_value, sensitivity, epsilon, rng):
    '''纯 ε-DP：加 Lap(0, b=sensitivity/epsilon) 噪声。'''
    b = sensitivity / epsilon
    return true_value + rng.laplace(0.0, b)

true_count = 42.0
sens, eps = 1.0, 0.5
b = sens / eps
draws = np.array([laplace_mechanism(true_count, sens, eps, rng) for _ in range(50000)])
print(f'真值={true_count}, ε={eps}, 噪声尺度 b={b}')
print(f'带噪均值 = {draws.mean():.3f}  (应≈真值 -> 无偏)')
print(f'带噪方差 = {draws.var():.3f}  (应≈2b²={2*b**2:.3f})')
assert abs(draws.mean() - true_count) < 0.1, 'Laplace 机制应无偏'
assert abs(draws.var() - 2*b**2) < 0.5, 'Laplace 方差应=2b²'
print('✅ Laplace 机制：无偏 + 方差=2b² 正确')

现在做**经验 ε 验证**——DP 保证的直接检验：

在一对相邻数据集（真值差 = 敏感度）上各跑机制很多次，估计输出落在某区间的概率比 `Pr[M(D)∈S] / Pr[M(D')∈S]`，验证它 **≤ e^ε**。

这就是隐私审计（模块 05）的雏形：用经验分布检查声称的隐私保证。

In [ ]:
def empirical_epsilon(value_D, value_Dp, sensitivity, epsilon, rng, n=400000, min_count=200):
    '''在相邻数据集(真值 value_D vs value_Dp)上估计经验隐私损失 = max |ln Pr比|。
       只在两边都有足够样本(>=min_count)的桶上估比值，避免尾部少样本桶的噪声虚高。'''
    a = np.array([laplace_mechanism(value_D,  sensitivity, epsilon, rng) for _ in range(n)])
    b = np.array([laplace_mechanism(value_Dp, sensitivity, epsilon, rng) for _ in range(n)])
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    edges = np.linspace(lo, hi, 60)
    ca, _ = np.histogram(a, bins=edges)
    cb, _ = np.histogram(b, bins=edges)
    mask = (ca >= min_count) & (cb >= min_count)        # 只看统计可靠的桶
    ratios = np.log((ca[mask] / n) / (cb[mask] / n))
    return float(np.max(np.abs(ratios)))

eps = 1.0
emp = empirical_epsilon(42.0, 43.0, sensitivity=1.0, epsilon=eps, rng=rng)
print(f'设定 ε = {eps},  e^ε = {np.exp(eps):.3f}')
print(f'经验隐私损失(可靠桶上 max|ln比值|) ≈ {emp:.3f}  (应 ≲ ε={eps})')
# 理论上 Laplace 机制在相邻真值间的对数密度比恰好 = Δ/b = ε；经验值应≈ε(留采样裕度)
assert emp <= eps * 1.25, '经验隐私损失不应超过设定 ε 太多（理论上应≈ε）'
print('✅ 经验验证：相邻数据集输出分布的对数比 ≈ ε —— DP 保证成立(这就是隐私审计的雏形)')

## 3 · Gaussian 机制：(ε,δ)-DP，配 L2 敏感度

加高斯噪声 `N(0, σ²)`，经典充分条件 `σ ≥ Δ₂·√(2 ln(1.25/δ)) / ε`。

它配 **L2 敏感度**，高维下比 Laplace 省噪声，且组合性质好——是 DP-SGD 的默认机制。验证无偏 + 方差正确。

In [ ]:
def gaussian_sigma(l2_sensitivity, epsilon, delta):
    '''经典(充分条件)高斯噪声标定。'''
    return l2_sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon

def gaussian_mechanism(true_value, l2_sensitivity, epsilon, delta, rng):
    sigma = gaussian_sigma(l2_sensitivity, epsilon, delta)
    return np.atleast_1d(true_value) + rng.normal(0.0, sigma, size=np.shape(np.atleast_1d(true_value)))

true_vec = np.array([10.0, -5.0, 3.0])
l2_sens, eps, delta = 1.0, 1.0, 1e-5
sigma = gaussian_sigma(l2_sens, eps, delta)
print(f'σ = {sigma:.3f}  (ε={eps}, δ={delta}, Δ₂={l2_sens})')
draws = np.array([gaussian_mechanism(true_vec, l2_sens, eps, delta, rng) for _ in range(20000)])
print(f'各维带噪均值 = {draws.mean(axis=0).round(3)}  (应≈{true_vec})')
print(f'各维带噪方差 = {draws.var(axis=0).round(3)}  (应≈σ²={sigma**2:.3f})')
assert np.allclose(draws.mean(axis=0), true_vec, atol=0.1), '高斯机制应无偏'
assert np.allclose(draws.var(axis=0), sigma**2, atol=0.5), '方差应≈σ²'
print('✅ Gaussian 机制：无偏 + 方差=σ² 正确')

**δ 越小 -> σ 越大；ε 越小 -> σ 越大**。我们扫一遍看 σ 怎么随 (ε,δ) 变化——这是后面标定 DP-SGD 噪声的基础。

In [ ]:
print(f"{'ε':>6s} {'δ':>10s} {'σ':>10s}")
base = gaussian_sigma(1.0, 1.0, 1e-5)
for eps in [0.5, 1.0, 2.0]:
    for delta in [1e-5, 1e-7]:
        s = gaussian_sigma(1.0, eps, delta)
        print(f'{eps:>6.1f} {delta:>10.0e} {s:>10.3f}')
# 单调性：ε 减半 -> σ 翻倍；δ 更小 -> σ 更大
assert gaussian_sigma(1.0,0.5,1e-5) > gaussian_sigma(1.0,1.0,1e-5), 'ε 越小 σ 越大'
assert gaussian_sigma(1.0,1.0,1e-7) > gaussian_sigma(1.0,1.0,1e-5), 'δ 越小 σ 越大'
print('✅ σ 随 ε 反比、随 δ 减小而增大 —— 隐私越强噪声越大')

## 4 · 组合定理：隐私预算如何累加

多次查询 = 多次花预算。三种核算：
- **基本组合**：k 次 ε → kε（线性）
- **高级组合**：≈ ε√(2k·ln(1/δ')) （√k，紧得多）
- **并行组合**：数据切不相交块各查 → 取 max（不是和）

我们实现三者并验证大小关系。

In [ ]:
def basic_composition(eps_each, k):
    return eps_each * k

def advanced_composition(eps_each, k, delta_prime):
    '''高级组合(常用形式)：总 ε ≈ √(2k ln(1/δ')) ε + k ε(e^ε-1)。'''
    return np.sqrt(2 * k * np.log(1/delta_prime)) * eps_each + k * eps_each * (np.exp(eps_each) - 1)

def parallel_composition(eps_list):
    '''不相交子集各查：总 ε = max（每个个体只落一个子集）。'''
    return max(eps_list)

eps_each, k, dp = 0.1, 100, 1e-6
basic = basic_composition(eps_each, k)
adv   = advanced_composition(eps_each, k, dp)
print(f'{k} 次 ε={eps_each} 查询:')
print(f'  基本组合 总ε = {basic:.3f}')
print(f'  高级组合 总ε = {adv:.3f}  (牺牲 δ\'={dp}, 但更紧)')
assert adv < basic, '高级组合应比基本组合紧（小 ε 多次时）'
# 并行：把数据切成 10 块各花 ε=0.1
par = parallel_composition([0.1]*10)
print(f'  并行组合(10块) 总ε = {par:.3f}  (取 max，不是和！)')
assert par == 0.1, '并行组合总 ε = 各块最大值'
print('✅ 高级组合(√k) < 基本(k)；并行(max) ≪ 串行(和)')

**后处理免疫**演示：对 DP 输出做任何不碰原始数据的处理，隐私保证不变。

我们对一个带噪计数做后处理（四舍五入、变换），它仍是同样 ε-DP 的——因为后处理不引入新的数据访问。

In [ ]:
# 带噪发布一个计数（ε=1 的 Laplace）
true_count = 137.0
noisy = laplace_mechanism(true_count, sensitivity=1.0, epsilon=1.0, rng=rng)

# 后处理：四舍五入到整数、截断到非负、再算个比例 —— 都不碰原始数据
post1 = max(0, round(noisy))
post2 = post1 / 1000.0
print(f'带噪计数 = {noisy:.3f} -> 四舍五入 {post1} -> 比例 {post2}')
print('这些后处理都只用了 noisy（已是 DP 输出），没有再读原始数据。')
# 形式化检验：后处理是 noisy 的确定性函数，不增加隐私损失
def is_post_processing_only(*, accessed_raw_data):
    return not accessed_raw_data
assert is_post_processing_only(accessed_raw_data=False), '真正的后处理不碰原始数据'
print('✅ 后处理免疫：DP 输出的任意后处理仍是同 ε-DP —— 隐私一旦买到就锁定')

## 5 · 隐私-效用权衡曲线

把「ε → 噪声 → 效用损失」整条曲线跑出来。对一个均值查询，扫不同 ε，看带噪结果的误差如何随隐私强度变化。

In [ ]:
def private_mean(data, clip, epsilon, rng):
    '''DP 均值：裁剪 -> 有界求和的 Laplace -> 除以 N。敏感度=clip/N。'''
    n = len(data)
    clipped = np.clip(data, 0, clip)
    sens = clip / n               # 均值的敏感度
    return laplace_mechanism(clipped.mean(), sens, epsilon, rng)

data = rng.uniform(0, 100, size=2000)
true_mean = np.clip(data, 0, 100).mean()
print(f'真实均值 = {true_mean:.3f}')
print(f"{'ε':>6s} {'平均|误差|':>12s}  {'隐私强度':<10s}")
prev = 1e9
for eps in [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]:
    errs = [abs(private_mean(data, 100, eps, rng) - true_mean) for _ in range(2000)]
    e = np.mean(errs)
    tag = '强' if eps <= 0.1 else ('中' if eps <= 1 else '弱')
    print(f'{eps:>6.2f} {e:>12.4f}  {tag:<10s}')
    assert e < prev, '误差应随 ε 增大(隐私变弱)而单调下降'
    prev = e
print('\n✅ ε 越大(越不私密) -> 误差越小(效用越好)。这条曲线是 DP 一切实践的中心。')

**大数据更友好**：均值敏感度 = clip/N 随 N 下降，所以同样 ε 下，数据越多误差越小。验证之。

In [ ]:
eps = 0.1
print(f"{'N':>8s} {'平均|误差|':>12s}")
prev = 1e9
for n in [100, 1000, 10000]:
    d = rng.uniform(0, 100, size=n)
    tm = np.clip(d, 0, 100).mean()
    e = np.mean([abs(private_mean(d, 100, eps, rng) - tm) for _ in range(1000)])
    print(f'{n:>8d} {e:>12.4f}')
    assert e < prev, '数据越多，同 ε 下误差应越小'
    prev = e
print('✅ 数据越多，同样隐私强度下效用越好 —— DP 在大数据上更友好')

---
## ✏️ 练习 1：实现 Gaussian 机制并标定噪声

实现 `make_gaussian_mechanism(l2_sensitivity, epsilon, delta)`，返回一个函数 `mech(true_value, rng)`，对输入加正确标定的高斯噪声（用经典充分条件 `σ = Δ₂√(2ln(1.25/δ))/ε`）。

In [ ]:
def make_gaussian_mechanism(l2_sensitivity, epsilon, delta):
    # TODO: 算出 σ，返回一个闭包 mech(true_value, rng)，
    #       对 np.atleast_1d(true_value) 加 N(0, σ²) 噪声并返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
mech = make_gaussian_mechanism(l2_sensitivity=2.0, epsilon=1.0, delta=1e-5)
exp_sigma = 2.0 * np.sqrt(2*np.log(1.25/1e-5)) / 1.0
draws = np.array([mech(np.array([0.0]), rng)[0] for _ in range(20000)])
assert abs(draws.mean()) < 0.2, '应无偏(均值≈0)'
assert abs(draws.std() - exp_sigma) < 0.5, f'噪声 std 应≈{exp_sigma:.2f}'
print(f'σ≈{draws.std():.2f} (期望 {exp_sigma:.2f})')
print('✅ 练习 1 通过：高斯机制噪声标定正确')

## ✏️ 练习 2：计算查询的敏感度

实现 `histogram_sensitivity(data, n_bins)`：一个**直方图**查询（每条记录落进一个桶，输出各桶计数向量）的 L1 敏感度。

提示：一条记录只影响一个桶（+1），所以增删一条记录使计数向量的 L1 变化恰好为 1。用 `empirical_l1_sensitivity` 验证你的解析答案。

In [ ]:
# 给定的直方图查询（已实现，供你分析其敏感度）
def histogram_query(data, n_bins):
    '''data 是 [0,1) 的值；分成 n_bins 个桶，返回计数向量。'''
    idx = np.minimum((data * n_bins).astype(int), n_bins - 1)
    return np.bincount(idx, minlength=n_bins).astype(float)

In [ ]:
def histogram_sensitivity(n_bins):
    # TODO: 返回直方图查询的解析 L1 敏感度（一个标量）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
data = rng.uniform(0, 1, size=200)
analytic = histogram_sensitivity(n_bins=10)
empirical = empirical_l1_sensitivity(lambda d: histogram_query(d, 10), data)
print(f'解析敏感度={analytic}, 经验敏感度={empirical}')
assert analytic == 1.0, '直方图 L1 敏感度应为 1'
assert abs(empirical - 1.0) < 1e-9, '经验敏感度应=1'
print('✅ 练习 2 通过：直方图敏感度=1 -> 可用并行组合，每桶独立加 Lap(1/ε)')

## ✏️ 练习 3：隐私预算核算

你要在一份数据上做 `k` 次查询，每次预算 `eps_each`，总预算上限 `eps_budget`。

实现 `max_queries(eps_each, eps_budget)`：用**基本组合**，最多能做几次查询而不超预算？（返回整数，向下取整。）

In [ ]:
def max_queries(eps_each, eps_budget):
    # TODO: 基本组合下 k*eps_each <= eps_budget 的最大整数 k
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert max_queries(0.1, 1.0) == 10, '1.0 预算、每次0.1 -> 10 次'
assert max_queries(0.3, 1.0) == 3, '向下取整'
assert max_queries(2.0, 1.0) == 0, '单次就超预算 -> 0 次'
k = max_queries(0.05, 1.0)
print(f'每次 ε=0.05、总预算 1.0 -> 最多 {k} 次查询')
assert k * 0.05 <= 1.0 + 1e-9 and (k+1)*0.05 > 1.0, '应是不超预算的最大次数'
print('✅ 练习 3 通过：会用基本组合管理隐私预算')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def make_gaussian_mechanism(l2_sensitivity, epsilon, delta):
    sigma = l2_sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon
    def mech(true_value, rng):
        v = np.atleast_1d(np.asarray(true_value, dtype=float))
        return v + rng.normal(0.0, sigma, size=v.shape)
    return mech

In [ ]:
# 练习 2 参考答案
def histogram_sensitivity(n_bins):
    return 1.0     # 一条记录只改一个桶 +1 -> L1 变化=1（与桶数无关）

In [ ]:
# 练习 3 参考答案
def max_queries(eps_each, eps_budget):
    # 注意浮点陷阱：1.0 // 0.1 == 9.0（0.1 无法精确表示），加微小容差修正
    return int(eps_budget / eps_each + 1e-9)

---
## 🧪 真实数据胶囊：复刻 Apple/Google 风格的 local DP 遥测统计

真实世界里，Apple 用 **local DP**（随机化回应）收集如「哪些 emoji / 网址最常用」的统计——用户在数据离开设备前就自己加噪，服务器只见带噪数据。

我们用**随机化回应（randomized response）**复刻它：估计「有多大比例的用户拥有某敏感属性」，每个用户都能否认自己的真实答案，服务器仍能无偏估计总体比例。这是 DP 最早、最经典的机制（Warner 1965）。

In [ ]:
def randomized_response(true_bit, p, rng):
    '''local DP：以概率 p 说真话，以概率 1-p 随机回答(0/1各半)。'''
    if rng.random() < p:
        return true_bit
    return int(rng.random() < 0.5)

def rr_epsilon(p):
    '''随机化回应的 ε：ln( Pr[报1|真1] / Pr[报1|真0] )。'''
    pr1_given1 = p + (1-p)*0.5
    pr1_given0 = (1-p)*0.5
    return np.log(pr1_given1 / pr1_given0)

def debias_estimate(responses, p):
    '''从带噪回答无偏估计真实比例：E[报1] = p*真比例 + (1-p)*0.5。'''
    obs = np.mean(responses)
    return (obs - (1-p)*0.5) / p

# 真实人群：30% 拥有某敏感属性
N = 100000
true_rate = 0.30
true_bits = (rng.random(N) < true_rate).astype(int)
p = 0.7                                   # 70% 概率说真话
responses = np.array([randomized_response(b, p, rng) for b in true_bits])
est = debias_estimate(responses, p)
eps = rr_epsilon(p)
print(f'真实比例 = {true_rate:.3f}')
print(f'去偏估计 = {est:.3f}  (从带噪回答恢复)')
print(f'本机制 ε = {eps:.3f}  (每个用户的 local DP 保证)')
assert abs(est - true_rate) < 0.02, '去偏估计应≈真实比例'
print('✅ 每个用户都能否认真实答案(local DP)，服务器仍无偏估计总体 —— 这就是 Apple/Google 遥测的内核')

**🧪 胶囊练习**：local DP 的代价是噪声更大。实现 `rr_std_error(p, true_rate, N)`：估计去偏估计量的标准误（多次模拟的 std），并验证 **p 越大（越不私密）-> ε 越大、标准误越小**（隐私换效用，在 local DP 里同样成立）。

In [ ]:
def rr_std_error(p, true_rate, N, trials=200, rng=None):
    # TODO: 模拟 trials 次：生成 N 个真 bit -> 随机化回应 -> 去偏估计；
    #       返回这些估计的标准差(std)
    raise NotImplementedError

In [ ]:
# 自测
rng2 = np.random.default_rng(1)
se_private = rr_std_error(0.55, 0.3, 5000, trials=100, rng=rng2)   # 更私密
se_loose   = rr_std_error(0.95, 0.3, 5000, trials=100, rng=rng2)   # 更不私密
print(f'p=0.55 (ε={rr_epsilon(0.55):.2f}) 标准误={se_private:.4f}')
print(f'p=0.95 (ε={rr_epsilon(0.95):.2f}) 标准误={se_loose:.4f}')
assert se_loose < se_private, '更不私密(p大,ε大) -> 标准误更小'
assert rr_epsilon(0.95) > rr_epsilon(0.55), 'p 越大 ε 越大'
print('✅ 胶囊练习通过：local DP 同样遵循隐私-效用权衡')

In [ ]:
# 📖 胶囊参考答案
def rr_std_error(p, true_rate, N, trials=200, rng=None):
    rng = rng or np.random.default_rng(0)
    ests = []
    for _ in range(trials):
        bits = (rng.random(N) < true_rate).astype(int)
        resp = np.where(rng.random(N) < p, bits, (rng.random(N) < 0.5).astype(int))
        ests.append((resp.mean() - (1-p)*0.5) / p)
    return float(np.std(ests))

### 小结
- **DP = 关于算法的、可证明的隐私**（取代脆弱的去标识/匿名化），对任意外部知识都成立。
- **(ε,δ)-DP**：相邻数据集输出分布的比 ≤ e^ε（+δ 松弛）；ε 是隐私预算，δ 取极小。
- **敏感度**决定噪声：Laplace 配 L1(纯ε-DP)、Gaussian 配 L2((ε,δ)-DP)；降敏感度靠裁剪。
- **组合**：串行加、并行取 max、高级组合 √k；**后处理免疫** = 隐私一旦买到就锁定。
- **隐私-效用权衡**是中心：没有免费的隐私，必须如实报告 ε 与代价。

下一站：**模块 02 · DP-SGD** —— 把 DP（尤其高斯机制 + 裁剪 + 组合）装进模型训练。